In [1]:
# show GPU + driver (run)
!nvidia-smi

Thu Sep 25 08:43:01 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print("torch.cuda.is_available() =", torch.cuda.is_available())
print("torch.cuda.device_count() =", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


torch.cuda.is_available() = True
torch.cuda.device_count() = 1
GPU name: Tesla T4


In [3]:
# basic python info
import sys
print("Python:", sys.version.splitlines()[0])

Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]


In [4]:
# Install CuPy if not present (common Colab runtime uses CUDA 11.x -> cupy-cuda11x)
try:
    import cupy as cp
    print("CuPy already installed, version:", cp.__version__)
except Exception:
    print("CuPy not found — installing cupy-cuda11x (may take ~1-2 minutes)...")
    !pip -q install cupy-cuda11x
    import importlib
    import cupy as cp
    importlib.invalidate_caches()
    print("Installed CuPy version:", cp.__version__)

CuPy already installed, version: 13.3.0


In [17]:
import cupy as cp
import numpy as np

# ---- CUDA kernel (naive matmul) ----
kernel_code = r'''
extern "C" __global__
void matmul(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        float val = 0.0f;
        for (int k = 0; k < N; ++k) {
            val += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = val;
    }
}
''';

# Compile kernel
module = cp.RawModule(code=kernel_code)
matmul_kernel = module.get_function("matmul")

# ---- Kernel runner ----
def run_kernel(A, B, C, block_dim):
    N = A.shape[0]
    bx, by = block_dim
    gx = (N + bx - 1) // bx
    gy = (N + by - 1) // by
    grid = (gx, gy)
    block = (bx, by)

    start, end = cp.cuda.Event(), cp.cuda.Event()
    start.record()
    matmul_kernel(grid, block, (A, B, C, np.int32(N)))
    end.record(); end.synchronize()
    return cp.cuda.get_elapsed_time(start, end) / 1000.0  # sec

# ---- Autotune ----
def autotune_matmul(N=512, configs=[(8,8), (16,16), (32,32)], trials=3):
    A = cp.random.randn(N, N, dtype=cp.float32)
    B = cp.random.randn(N, N, dtype=cp.float32)
    C = cp.zeros((N, N), dtype=cp.float32)

    # Reference with cuBLAS
    start, end = cp.cuda.Event(), cp.cuda.Event()
    start.record()
    ref = A.dot(B)
    end.record(); end.synchronize()
    ref_time = cp.cuda.get_elapsed_time(start, end) / 1000.0

    best_cfg, best_time = None, float("inf")
    print(f"\nAutotuning N={N}...\n")
    for cfg in configs:
        times = [run_kernel(A, B, C, cfg) for _ in range(trials)]
        tmin = min(times)
        print(f"Block {cfg}: {times} -> best {tmin:.6f}s")
        if tmin < best_time:
            best_cfg, best_time = cfg, tmin

    # Numerical correctness check
    max_err = float(cp.max(cp.abs(C - ref)))
    print(f"\ncuBLAS time: {ref_time:.6f}s")
    print(f"Best block {best_cfg}, time {best_time:.6f}s, error {max_err:.3e}")

if __name__ == "__main__":
    autotune_matmul(N=512, configs=[(8,8), (16,16), (32,32)])



Autotuning N=512...

Block (8, 8): [0.0019052480459213258, 0.0018718080520629882, 0.0018486080169677734] -> best 0.001849s
Block (16, 16): [0.0011855360269546508, 0.0011755839586257936, 0.0011648319959640502] -> best 0.001165s
Block (32, 32): [0.0009056000113487244, 0.000899071991443634, 0.0008972160220146179] -> best 0.000897s

cuBLAS time: 0.000371s
Best block (32, 32), time 0.000897s, error 0.000e+00


In [18]:
import cupy as cp
import numpy as np

# ---- CUDA kernel (naive matmul) ----
kernel_code = r'''
extern "C" __global__
void matmul(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        float val = 0.0f;
        for (int k = 0; k < N; ++k) {
            val += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = val;
    }
}
''';

# Compile kernel
module = cp.RawModule(code=kernel_code)
matmul_kernel = module.get_function("matmul")

# ---- Kernel runner ----
def run_kernel(A, B, C, block_dim):
    N = A.shape[0]
    bx, by = block_dim
    gx = (N + bx - 1) // bx
    gy = (N + by - 1) // by
    grid = (gx, gy)
    block = (bx, by)

    start, end = cp.cuda.Event(), cp.cuda.Event()
    start.record()
    matmul_kernel(grid, block, (A, B, C, np.int32(N)))
    end.record(); end.synchronize()
    return cp.cuda.get_elapsed_time(start, end) / 1000.0  # sec

# ---- Autotune ----
def autotune_matmul(N=512, configs=[(8,8), (16,16), (32,32)], trials=3):
    A = cp.random.randn(N, N, dtype=cp.float32)
    B = cp.random.randn(N, N, dtype=cp.float32)
    C = cp.zeros((N, N), dtype=cp.float32)

    # Reference with cuBLAS
    start, end = cp.cuda.Event(), cp.cuda.Event()
    start.record()
    ref = A.dot(B)
    end.record(); end.synchronize()
    ref_time = cp.cuda.get_elapsed_time(start, end) / 1000.0

    best_cfg, best_time = None, float("inf")
    print(f"\nAutotuning N={N}...\n")
    for cfg in configs:
        times = [run_kernel(A, B, C, cfg) for _ in range(trials)]
        tmin = min(times)
        print(f"Block {cfg}: {times} -> best {tmin:.6f}s")
        if tmin < best_time:
            best_cfg, best_time = cfg, tmin

    # Numerical correctness check
    max_err = float(cp.max(cp.abs(C - ref)))
    print(f"\ncuBLAS time: {ref_time:.6f}s")
    print(f"Best block {best_cfg}, time {best_time:.6f}s, error {max_err:.3e}")

if __name__ == "__main__":
    autotune_matmul(N=512, configs=[(8,8), (16,16), (32,32)])




Autotuning N=512...

Block (8, 8): [0.0020056960582733154, 0.001882048010826111, 0.001880095958709717] -> best 0.001880s
Block (16, 16): [0.0011837760210037232, 0.0011832000017166138, 0.0011946560144424439] -> best 0.001183s
Block (32, 32): [0.0009170879721641541, 0.000918944001197815, 0.0009175360202789307] -> best 0.000917s

cuBLAS time: 0.000346s
Best block (32, 32), time 0.000917s, error 0.000e+00


In [19]:
import cupy as cp
import numpy as np

# ---- CUDA kernel (naive matmul) ----
kernel_code = r'''
extern "C" __global__
void matmul(const float* A, const float* B, float* C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        float val = 0.0f;
        for (int k = 0; k < N; ++k) {
            val += A[row * N + k] * B[k * N + col];
        }
        C[row * N + col] = val;
    }
}
''';

# Compile kernel
module = cp.RawModule(code=kernel_code)
matmul_kernel = module.get_function("matmul")

# ---- Kernel runner ----
def run_kernel(A, B, C, block_dim):
    N = A.shape[0]
    bx, by = block_dim
    gx = (N + bx - 1) // bx
    gy = (N + by - 1) // by
    grid = (gx, gy)
    block = (bx, by)

    start, end = cp.cuda.Event(), cp.cuda.Event()
    start.record()
    matmul_kernel(grid, block, (A, B, C, np.int32(N)))
    end.record(); end.synchronize()
    return cp.cuda.get_elapsed_time(start, end) / 1000.0  # seconds

# ---- Benchmark ----
def benchmark_matmul(N=512, configs=[(8,8), (16,16), (32,32)], trials=5):
    A = cp.random.randn(N, N, dtype=cp.float32)
    B = cp.random.randn(N, N, dtype=cp.float32)
    C = cp.zeros((N, N), dtype=cp.float32)

    # Reference with cuBLAS
    start, end = cp.cuda.Event(), cp.cuda.Event()
    start.record()
    ref = A.dot(B)
    end.record(); end.synchronize()
    ref_time = cp.cuda.get_elapsed_time(start, end) / 1000.0

    print(f"\nBenchmarking N={N} with configs {configs}...\n")
    results = {}
    for cfg in configs:
        times = [run_kernel(A, B, C, cfg) for _ in range(trials)]
        avg_time = sum(times) / len(times)
        best_time = min(times)
        results[cfg] = (avg_time, best_time)
        print(f"Block {cfg}: times={['%.6f'%t for t in times]} "
              f"-> avg={avg_time:.6f}s, best={best_time:.6f}s")

    # Numerical correctness check
    max_err = float(cp.max(cp.abs(C - ref)))
    print(f"\ncuBLAS (A.dot(B)) time: {ref_time:.6f}s")
    print(f"Max error vs cuBLAS: {max_err:.3e}")

    # Find best by average time
    best_cfg = min(results.keys(), key=lambda k: results[k][0])
    print(f"\n=> Best config by AVERAGE time: {best_cfg} "
          f"({results[best_cfg][0]:.6f}s avg)")

    return results

if __name__ == "__main__":
    benchmark_matmul(N=512, configs=[(8,8), (16,16), (32,32)], trials=5)



Benchmarking N=512 with configs [(8, 8), (16, 16), (32, 32)]...

Block (8, 8): times=['0.001907', '0.001877', '0.001878', '0.001868', '0.001861'] -> avg=0.001878s, best=0.001861s
Block (16, 16): times=['0.001184', '0.001183', '0.001182', '0.001183', '0.001182'] -> avg=0.001183s, best=0.001182s
Block (32, 32): times=['0.000918', '0.000918', '0.000917', '0.000918', '0.000917'] -> avg=0.000918s, best=0.000917s

cuBLAS (A.dot(B)) time: 0.000532s
Max error vs cuBLAS: 0.000e+00

=> Best config by AVERAGE time: (32, 32) (0.000918s avg)
